<a href="https://colab.research.google.com/github/adi542/ai-ml/blob/main/text_classification_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import list_datasets
all_datasets = list(list_datasets(limit=100))
print(len(all_datasets))

In [ ]:
print(all_datasets[:10])

In [ ]:
from datasets import load_dataset

In [ ]:
emotion = load_dataset("dair-ai/emotion")

In [ ]:
emotion

In [ ]:
train_ds = emotion["train"]
test_ds = emotion["test"]
val_ds = emotion["validation"]



In [ ]:
len(train_ds), len(test_ds), len(val_ds)

In [ ]:
import pandas as pd

In [ ]:
emotion.set_format(type="pandas")
df = emotion["train"][:]
df.head()

In [ ]:
def label_int2str(row):
  return emotion["train"].features["label"].int2str(row)

new column is added in the table,for all the row of the label the function label_int2str is applied for each row with each label value

In [ ]:
df["label_name"] = df["label"].apply(label_int2str)
df.head()

In [ ]:
import matplotlib.pyplot as plt
df["label_name"].value_counts(ascending=True).plot.barh()
plt.title("Frequency of emotion")
plt.show()

In [ ]:
df["words per tweet"] = df["text"].str.split().apply(len)

In [ ]:
df.head()

In [ ]:
from transformers import AutoTokenizer


In [ ]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


In [ ]:
text = "hello my name is aditya kochhar"

In [ ]:
encoded_text = tokenizer(text)
encoded_text

In [ ]:
token = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
token

In [ ]:
emotion.reset_format()

In [ ]:
def tokenize(batch):
  return tokenizer(batch["text"], padding=True, truncation=True)

In [ ]:
print(tokenize(emotion["train"][:2]))

In [ ]:
print(len(emotion["train"]))

In [ ]:
emotion_encoded = emotion.map(tokenize, batched=True, batch_size=None)
print(emotion_encoded["train"].column_names)




In [ ]:
import torch
from torch import nn

In [ ]:
torch.cuda.is_available()

In [ ]:
from transformers import AutoModel
model_ckpt = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_ckpt).to(device)

In [ ]:
text = "this is a test"
inputs = tokenizer(text,return_tensors="pt").to(device)
print(inputs)

In [ ]:
print(inputs['input_ids'].shape)

In [ ]:
new_inputs = {}
for k,v in inputs.items():
  new_inputs[k] = v.to(device)

In [ ]:
new_inputs

In [ ]:
with torch.no_grad():
  outputs = model(**inputs)
print(outputs)

In [ ]:
outputs.last_hidden_state.size()

In [ ]:
outputs.last_hidden_state[:,0,:].size()

In [ ]:
import torch

def extract_hidden_states(batch):
  inputs = {
        k: torch.tensor(v).to(device)
        for k, v in batch.items()
        if k in ["input_ids", "attention_mask"]
    }
  with torch.no_grad():
    outputs = model(**inputs).last_hidden_state[:,0,:].cpu().numpy()

  return {"hidden_states": outputs}

In [ ]:
emotion_encoded.reset_format()

In [ ]:
emotion_encoded.set_format("torch",columns=["input_ids","attention_mask","label"])

In [ ]:
!pip install -U torch torchvision datasets


In [ ]:
emotions_hidden = emotion_encoded.map(extract_hidden_states, batched=True)

In [ ]:
emotions_hidden["train"].column_names

In [ ]:
device

In [ ]:
import numpy as np

In [ ]:
X_train = np.array(emotions_hidden["train"]["hidden_states"])
X_valid = np.array(emotions_hidden["validation"]["hidden_states"])
y_train = np.array(emotions_hidden["train"]["label"])
y_valid = np.array(emotions_hidden["validation"]["label"])
X_train.shape, X_valid.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
lr_clf = LogisticRegression(max_iter=3000)
lr_clf.fit(X_train, y_train)
lr_clf.score(X_valid, y_valid)

In [ ]:
from sklearn.dummy import DummyClassifier
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
dummy_clf.score(X_valid, y_valid)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay,confusion_matrix


In [ ]:
def plot_confusion_matrix(y_preds, y_true,labels):
  cm = confusion_matrix(y_true, y_preds,normalize='true')
  fig,ax = plt.subplots(figsize=(6,6))
  disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=labels)
  disp.plot(cmap="Blues",values_format=".2f",ax=ax,colorbar=False)
  plt.title("Normalized confusion matrix")
  plt.show()

In [ ]:
y_preds = lr_clf.predict(X_valid)

In [ ]:
plot_confusion_matrix(y_preds,y_valid,emotions_hidden["train"].features["label"].names)


we will get the sequnceClassification model from hugging face that have the classificatian head on top

In [ ]:
model_ckpt

In [ ]:
from transformers import AutoModelForSequenceClassification
num_label = 6
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt,num_labels=num_label).to(device)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  f1 = f1_score(labels,preds,average="weighted")
  acc = accuracy_score(labels,preds)
  return {"accuracy" : acc,"f1":f1}

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
print("hello")

In [ ]:
from transformers import TrainingArguments,Trainer
batch_size = 64
logging_steps = len(emotions_hidden["train"]) // batch_size
model_name = f"{model_ckpt}-finetuned-emotion"
training_args = TrainingArguments(output_dir=model_name,num_train_epochs=2,
                                  learning_rate=2e-5,per_device_train_batch_size=batch_size,per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,disable_tqdm=False,logging_steps=logging_steps,push_to_hub=True,
                                  log_level="error",eval_strategy="epoch")

In [ ]:
from transformers import Trainer

trainer = Trainer(model=model,args=training_args,compute_metrics=compute_metrics,
                  train_dataset=emotion_encoded["train"],
                  eval_dataset=emotion_encoded["validation"],
                  processing_class=tokenizer)

In [ ]:
trainer.train();

In [ ]:
pred_output = trainer.predict(emotion_encoded["validation"])

In [ ]:
pred_output

In [ ]:
y_preds = np.argmax(pred_output.predictions,axis=1)

In [ ]:
plot_confusion_matrix(y_preds,y_valid,emotions_hidden["train"].features["label"].names)

In [ ]:
import time

In [ ]:
trainer.push_to_hub(commit_message="Training completed")

In [ ]:
from transformers import pipeline



In [ ]:
model_id = "UnknownPro123/distilbert-base-uncased-finetuned-emotion"
classifier = pipeline("text-classification",model=model_id)

In [ ]:
classifier.model.config.id2label = {
    0:'sadness',
    1:'joy',
    2:'love',
    3:'anger',
    4:'fear',
    5:'surprise'
}

In [ ]:
custom_text = "i saw a movie today and it was really good"
preds = classifier(custom_text)
print(preds)

In [ ]:
classifier.model.config.id2label